# Llama-3.2-1B-Instruct

In [ ]:
# Install dependencies
!pip install transformers datasets peft accelerate scikit-learn              matplotlib seaborn bitsandbytes -q


In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, auc, classification_report
)

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

# ── Configuration ──────────────────────────────────────────────────────────
MODEL_ID    = "unsloth/Llama-3.2-1B-Instruct"
HF_TOKEN    = ""                     # Set your HuggingFace token here
MAX_LEN     = 256
TRAIN_SEED  = 42
TRAIN_SIZE  = 1000                   # Subsample size to keep training feasible
VALID_SIZE  = 250
BATCH_SIZE  = 2
GRAD_ACC    = 16
LR          = 2e-4
EPOCHS      = 1
LORA_R      = 16
LORA_ALPHA  = 32
LORA_DROP   = 0.1
NUM_LABELS  = 2
OUTPUT_DIR  = f"./results-llama-3.2-1b-instruct"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
print(f"Model  : {MODEL_ID}")


In [ ]:
# Log in to HuggingFace (required for gated models)
from huggingface_hub import login
if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    print("WARNING: HF_TOKEN not set. Gated models will fail to load.")


In [ ]:
# Load and subsample dataset
dataset = load_dataset("google/code_x_glue_cc_defect_detection")

train_full = dataset["train"].shuffle(seed=TRAIN_SEED)
valid_full = dataset["validation"].shuffle(seed=TRAIN_SEED)
test_split = dataset["test"]

train_sub = train_full.select(range(min(TRAIN_SIZE, len(train_full))))
valid_sub = valid_full.select(range(min(VALID_SIZE, len(valid_full))))

print(f"Train subset : {len(train_sub):,}")
print(f"Valid subset : {len(valid_sub):,}")
print(f"Test split   : {len(test_split):,}")

# Label check
import collections
print("Train label dist:", collections.Counter(train_sub["target"]))


In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN if HF_TOKEN else None,
    trust_remote_code=True,
)

# Decoder-only models often lack a pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("Vocab size :", tokenizer.vocab_size)
print("Pad token  :", tokenizer.pad_token)


In [ ]:
# Tokenize datasets
def tokenize(batch):
    tokens = tokenizer(
        batch["func"],
        truncation=True,
        max_length=MAX_LEN,
        padding=False,      # DataCollatorWithPadding handles dynamic padding
    )
    tokens["labels"] = [int(label) for label in batch["target"]]
    return tokens

cols_to_remove = ["func", "target", "project", "commit_id"]

train_tok = train_sub.map(tokenize, batched=True, remove_columns=cols_to_remove)
valid_tok = valid_sub.map(tokenize, batched=True, remove_columns=cols_to_remove)
test_tok  = test_split.map(tokenize, batched=True, remove_columns=cols_to_remove)

train_tok.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
valid_tok.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_tok.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

print("Tokenization complete.")
print("Train features:", train_tok.features)
print("Label dtype sample:", train_tok[0]["labels"].dtype)


In [ ]:
# Load base model with classification head
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=NUM_LABELS,
    token=HF_TOKEN if HF_TOKEN else None,
    trust_remote_code=True,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None,
    ignore_mismatched_sizes=True,
)

# Align classifier config
model.config.pad_token_id = tokenizer.pad_token_id
model.config.problem_type = "single_label_classification"

# Apply LoRA (PEFT)
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROP,
    bias="none",
    # Target standard attention projection layers
    target_modules=["q_proj", "v_proj"],
)

model = get_peft_model(model, lora_config)

# Keep trainable modules in the model dtype to avoid classifier-head dtype mismatches.
model.print_trainable_parameters()


In [ ]:
# Compute metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy":  accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall":    recall_score(labels, preds, zero_division=0),
        "f1":        f1_score(labels, preds, zero_division=0),
    }


In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACC,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.06,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=False,                  # Disabled: LoRA trainable params are float32
    logging_steps=50,
    report_to="none",
    dataloader_num_workers=2,
    seed=TRAIN_SEED,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=valid_tok,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Trainer initialized.")


In [ ]:
# Train the model
train_result = trainer.train()
print("Training complete.")
print(f"Training loss     : {train_result.training_loss:.4f}")
print(f"Training runtime  : {train_result.metrics['train_runtime']:.1f}s")


In [ ]:
# Plot training loss curve
log_history = trainer.state.log_history
train_logs = [x for x in log_history if "loss" in x and "eval_loss" not in x]
eval_logs  = [x for x in log_history if "eval_loss" in x]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

if train_logs:
    steps = [x["step"] for x in train_logs]
    losses = [x["loss"] for x in train_logs]
    axes[0].plot(steps, losses, color="#4C72B0", linewidth=1.5)
    axes[0].set_xlabel("Step")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("Training Loss", fontweight="bold")
    axes[0].spines[["top", "right"]].set_visible(False)

if eval_logs:
    epochs_e = [x["epoch"] for x in eval_logs]
    f1_scores = [x.get("eval_f1", 0) for x in eval_logs]
    axes[1].plot(epochs_e, f1_scores, marker="o", color="#DD8452", linewidth=2)
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("F1 Score")
    axes[1].set_title("Validation F1 per Epoch", fontweight="bold")
    axes[1].spines[["top", "right"]].set_visible(False)
    axes[1].set_ylim(0, 1)

plt.suptitle(f"Training Curves", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("training_curves.png", bbox_inches="tight")
plt.show()


In [ ]:
# Evaluate on test set
test_output = trainer.predict(test_tok)
logits = test_output.predictions
labels = test_output.label_ids
probs  = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
preds  = np.argmax(logits, axis=-1)

acc = accuracy_score(labels, preds)
pre = precision_score(labels, preds, zero_division=0)
rec = recall_score(labels, preds, zero_division=0)
f1  = f1_score(labels, preds, zero_division=0)
fpr, tpr, _ = roc_curve(labels, probs)
roc_auc = auc(fpr, tpr)

print("=== Test Set Evaluation ===")
print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {pre:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"AUC       : {roc_auc:.4f}")
print()
print(classification_report(labels, preds, target_names=["Clean", "Defective"]))


In [ ]:
# Visualization: Confusion Matrix + ROC Curve
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion Matrix
cm = confusion_matrix(labels, preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Clean", "Defective"],
            yticklabels=["Clean", "Defective"],
            ax=axes[0], linewidths=0.5)
axes[0].set_xlabel("Predicted Label", fontweight="bold")
axes[0].set_ylabel("True Label", fontweight="bold")
axes[0].set_title("Confusion Matrix (Test Set)", fontweight="bold")

# ROC Curve
axes[1].plot(fpr, tpr, color="#4C72B0", linewidth=2,
             label=f"AUC = {roc_auc:.4f}")
axes[1].plot([0, 1], [0, 1], "k--", linewidth=1, label="Random")
axes[1].fill_between(fpr, tpr, alpha=0.1, color="#4C72B0")
axes[1].set_xlabel("False Positive Rate", fontweight="bold")
axes[1].set_ylabel("True Positive Rate", fontweight="bold")
axes[1].set_title("ROC Curve (Test Set)", fontweight="bold")
axes[1].legend(loc="lower right")
axes[1].spines[["top", "right"]].set_visible(False)

plt.suptitle(f"Evaluation Results", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("evaluation_results.png", bbox_inches="tight")
plt.show()


In [ ]:
# Per-class metrics bar chart
class_report = classification_report(
    labels, preds,
    target_names=["Clean", "Defective"],
    output_dict=True
)

metrics = ["precision", "recall", "f1-score"]
x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width/2,
               [class_report["Clean"][m] for m in metrics],
               width, label="Clean", color="#4C72B0", edgecolor="white")
bars2 = ax.bar(x + width/2,
               [class_report["Defective"][m] for m in metrics],
               width, label="Defective", color="#DD8452", edgecolor="white")

for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f"{bar.get_height():.3f}",
            ha="center", va="bottom", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(["Precision", "Recall", "F1-Score"])
ax.set_ylim(0, 1.12)
ax.set_ylabel("Score")
ax.set_title("Per-Class Metrics (Test Set)", fontweight="bold")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("per_class_metrics.png", bbox_inches="tight")
plt.show()


In [ ]:
# Score distribution by true label
fig, ax = plt.subplots(figsize=(9, 4))
for label_val, color, name in zip([0, 1], ["#4C72B0", "#DD8452"], ["Clean", "Defective"]):
    subset_probs = probs[labels == label_val]
    ax.hist(subset_probs, bins=40, alpha=0.65, color=color,
            label=name, edgecolor="none")
ax.axvline(0.5, color="red", linestyle="--", linewidth=1.5, label="Decision boundary")
ax.set_xlabel("Predicted Probability (Defective)")
ax.set_ylabel("Count")
ax.set_title("Prediction Score Distribution by True Label", fontweight="bold")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("score_distribution.png", bbox_inches="tight")
plt.show()


In [ ]:
# Final results summary
print("=== Final Results Summary ===")
results_df = pd.DataFrame([{
    "Model": MODEL_ID,
    "Accuracy":  round(acc, 4),
    "Precision": round(pre, 4),
    "Recall":    round(rec, 4),
    "F1":        round(f1, 4),
    "AUC":       round(roc_auc, 4),
}])
print(results_df.to_string(index=False))


In [ ]:
# Save the fine-tuned model
model.save_pretrained(OUTPUT_DIR + "/best_model")
tokenizer.save_pretrained(OUTPUT_DIR + "/best_model")
print(f"Model saved to: {OUTPUT_DIR}/best_model")
